In [1]:
from datetime import datetime
from presetgrids import *
from numba import jit
import numpy as np
from utils import *

# Define constants for possible corner directions
DIR_NE = 0
DIR_NW = 1
DIR_SE = 2
DIR_SW = 3

X = 1
O = 2

In [2]:
'''
    Functions with the @jit decorator are compiled by Numba and run directly as machine code, bypassing the usual Python interpreter. This makes "jitted" (JIT stands for just in time)
    functions 1-2 orders of magnitude faster than non-jitted functions. The trade-off is that jitted functions can only call Numpy functions, 
    predefined Python functions, and other jitted functions. There are other restrictions as well, see https://numba.readthedocs.io/en/stable/user/jit.html
'''

'''
    This function returns the first occurrence of a value "val" in an array "arr" , for -1 if the value is not found.
'''
@jit(nopython=True)
def first_occurrence(arr, val):
    for i in range(len(arr)):
        if(arr[i] == val):
            return i
    return -1

'''
    This function adds "value" to each element of "arr" that is greater than or equal to "min_threshold"
'''
@jit(nopython=True)
def increment_if_geq(arr, min_threshold, value):
    for i in range(len(arr)):
        if(arr[i] >= min_threshold):
            arr[i] += value

'''
    This function counts the number of + and - crossings in a grid, given its "X_by_row" and "O_by_row" arrays, along with its height/width size "n"
'''
@jit(nopython=True)
def count_crossings(X_by_row, O_by_row, n):
    pos = 0
    neg = 0
    for row in range(n):
        x_marking, o_marking = X_by_row[row], O_by_row[row]
        min_col = min(x_marking, o_marking)
        cols_interval = [min_col, x_marking + o_marking - min_col]

        for col in range(cols_interval[0], cols_interval[1]):
            x_row_idx = first_occurrence(X_by_row, col)
            o_row_idx = first_occurrence(O_by_row, col)
            min_row = min(x_row_idx, o_row_idx)
            rows_interval = [min_row, x_row_idx + o_row_idx - min_row]

            if(row > rows_interval[0] and row < rows_interval[1]): # crossing detected
                if(min_col == x_marking): # X           O in the row
                    if(min_row == x_row_idx): # X            O  in the col
                        neg += 1
                    else:
                        pos += 1
                else: # O         X  in the row
                    if(min_row == x_row_idx): # X            O  in the col
                        pos += 1
                    else:
                        neg += 1
                        
    return pos, neg  

'''
    This returns the direction of a corner at coordinates ("row", "col")
'''
@jit(nopython=True)
def get_corner_dir(X_by_row, O_by_row, row, col, n):
    assert(row < n and col < n)
    if(X_by_row[row] == col):
        if(O_by_row[row] < col and first_occurrence(O_by_row, col) > row):
            return DIR_NE
        if(O_by_row[row] > col and first_occurrence(O_by_row, col) > row):
            return DIR_NW
        if(O_by_row[row] < col and first_occurrence(O_by_row, col) < row):
            return DIR_SE
        if(O_by_row[row] > col and first_occurrence(O_by_row, col) < row):
            return DIR_SW
    elif(O_by_row[row] == col):
        if(X_by_row[row] < col and first_occurrence(X_by_row, col) > row):
            return DIR_NE
        if(X_by_row[row] > col and first_occurrence(X_by_row, col) > row):
            return DIR_NW
        if(X_by_row[row] < col and first_occurrence(X_by_row, col) < row):
            return DIR_SE
        if(X_by_row[row] > col and first_occurrence(X_by_row, col) < row):
            return DIR_SW
    else:
        raise NotImplementedError

'''
    This function performs stabilization on a grid.
'''
@jit(nopython=True)
def stabilization_worker(X_by_row, O_by_row, row, col, letter, dir, n, row_olm_coords, col_olm_coords):
        # Step 1: Find locations of the other letter along the row and column of stabilization

        # If the letter at the stabilization location is 'X', then the "other_letter" is 'O', and vice versa.
        if(letter == X):
            same_letter_dict = X_by_row
            other_letter_dict = O_by_row
        else:
            same_letter_dict = O_by_row
            other_letter_dict = X_by_row

        # Find the indices of the other_letter in the row an column. 
        ol_idx_in_col = first_occurrence(other_letter_dict, col)
        ol_idx_in_row = other_letter_dict[row]

        # Put them in coordinate form
        col_olm_coords = [ol_idx_in_col, col]
        row_olm_coords = [row, ol_idx_in_row]

        # We enlarge the grid by increasing coordinate values after the location of stabilization
        increment_if_geq(X_by_row, col+1, 1)
        increment_if_geq(O_by_row, col+1, 1)
        
        X_by_row[row+2: n+1] = X_by_row[row+1 : n]
        O_by_row[row+2: n+1] = O_by_row[row+1 : n]
        
        '''
            We fill in the empty 2x2 square in the grid with three new markings. For example
            X:NE would be

            X 
            OX
        '''
        if(dir == DIR_NE):
            same_letter_dict[row] = col
            same_letter_dict[row+1] = col+1
            other_letter_dict[row+1] = col

            offset_row = 0
            offset_col = 1
        elif(dir == DIR_NW):
            same_letter_dict[row] = col+1
            same_letter_dict[row+1] = col
            other_letter_dict[row+1] = col+1

            offset_row = 0
            offset_col = 0
        elif(dir == DIR_SE):
            same_letter_dict[row] = col+1
            same_letter_dict[row+1] = col
            other_letter_dict[row] = col

            offset_row = 1
            offset_col = 1
        else: 
            same_letter_dict[row] = col
            same_letter_dict[row+1] = col+1
            other_letter_dict[row] = col+1

            offset_row = 1
            offset_col = 0

        # Add back missing markings in the row, col
        col_olm_coords[1] += offset_col
        row_olm_coords[0] += offset_row
        if(col_olm_coords[0] > row):
            col_olm_coords[0] += 1
        if(row_olm_coords[1] > col):
            row_olm_coords[1] += 1
        
        other_letter_dict[col_olm_coords[0]] = col_olm_coords[1]
        other_letter_dict[row_olm_coords[0]] = row_olm_coords[1]

# Perform a row commutation
@jit(nopython=True)
def row_commutation(X_by_row, O_by_row, first_row_idx, n):
    if(first_row_idx < 0 or first_row_idx >= n-1):
        raise NotImplementedError
    
    # Determine if the markings of one row are "sandwiched" between another or if they are disjoint
    a_l = [X_by_row[first_row_idx], O_by_row[first_row_idx]]
    a_l.sort()
    b_l = [X_by_row[first_row_idx+1], O_by_row[first_row_idx+1]]
    b_l.sort()

    # a_1 = sorted length-2 array of the column indices of the letters in the first row
    # a_2 the same but for the second row
    a_1 = a_l[0]
    a_2 = a_l[1]
    b_1 = b_l[0]
    b_2 = b_l[1]
    assert(a_1 < a_2)
    assert(b_1 < b_2)
    
    # The first two cases are when one is nested in another. The last two are when they are disjoint.
    if((a_1 < b_1 and a_2 > b_2) or (a_1 > b_1 and a_2 < b_2) or (a_2 < b_1) or (b_2 < a_1)):
        # Is possible
        X_by_row[first_row_idx], X_by_row[first_row_idx+1] = X_by_row[first_row_idx+1], X_by_row[first_row_idx]
        O_by_row[first_row_idx], O_by_row[first_row_idx+1] = O_by_row[first_row_idx+1], O_by_row[first_row_idx]

    else: # Cannot
        raise NotImplementedError

# Column commutation
@jit(nopython=True)
def column_commutation(X_by_row, O_by_row, first_col_idx, n):
    # We assume it is permitted
    
    if(first_col_idx < 0 or first_col_idx >= n-1):
        raise NotImplementedError
    
    X_row_coord_firstcol = first_occurrence(X_by_row, first_col_idx)
    X_row_coord_secondcol = first_occurrence(X_by_row, first_col_idx+1)
    X_by_row[X_row_coord_firstcol] = first_col_idx+1
    X_by_row[X_row_coord_secondcol] = first_col_idx

    O_row_coord_firstcol = first_occurrence(O_by_row, first_col_idx)
    O_row_coord_secondcol = first_occurrence(O_by_row, first_col_idx+1)
    O_by_row[O_row_coord_firstcol] = first_col_idx+1
    O_by_row[O_row_coord_secondcol] = first_col_idx

In [3]:
class GridDiagram:
    def __init__(self, gridsize, gridsize_max=15):
        self.n_max = gridsize_max # The maximum possible grid size. It is not possible to enlarge the grid if it reaches this size
        self.n = gridsize

        self.X_by_row = np.full(gridsize_max, -1) # An array s.t. X_by_row[i] is the column index of the 'X' marking in ith row.
        self.O_by_row = np.full(gridsize_max, -1) # An array s.t. O_by_row[i] is the column index of the 'O' marking in ith row.

    # Count the number of components
    def count_components(self):
        all_seen_X_rows = []
        components = 0
        cur_start_X_row = 0
        while(True):
            seen_X_rows = []
            while(True):
                if(cur_start_X_row in all_seen_X_rows):
                    cur_start_X_row += 1
                    if(cur_start_X_row == self.n):
                        return components
                else:
                    break
            
            cur_row = cur_start_X_row
            while(True):
                seen_X_rows.append(cur_row)
                all_seen_X_rows.append(cur_row)
                cur_row = first_occurrence(self.X_by_row, self.O_by_row[cur_row])
                if(cur_row in seen_X_rows):
                    components += 1
                    break

    # Count the number of up and down cusps                
    def count_cusps(self):
        up = 0
        down = 0

        for row, col in enumerate(self.X_by_row[:self.n]):
            dir = get_corner_dir(self.X_by_row, self.O_by_row, row, col, self.n)
            if(dir == DIR_NW):
                down += 1
            elif(dir == DIR_SE):
                up += 1
        
        for row, col in enumerate(self.O_by_row[:self.n]):
            dir = get_corner_dir(self.X_by_row, self.O_by_row, row, col, self.n)
            if(dir == DIR_NW):
                up += 1
            elif(dir == DIR_SE):
                down += 1
        
        return up, down
    
    def writhe(self):
        pos, neg = count_crossings(self.X_by_row, self.O_by_row, self.n)
        return pos - neg

    def tb(self):
        return self.writhe() - sum(self.count_cusps())/2
    
    def r(self):
        up, down = self.count_cusps()
        return (down - up)/2
    
    # A function to check that the X_by_row and O_by_row arrays are valid. Used for debugging
    def validate(self):
        if(set(self.X_by_row) != set(range(-1, self.n))):
            return -1
        if(set(self.O_by_row) != set(range(-1, self.n))):
            return -2
        if(np.count_nonzero(self.X_by_row != -1) != self.n):
            return -3
        if(np.count_nonzero(self.O_by_row != -1) != self.n):
            return -4
        return 0

    def convert_to_grid(self):
        grid = np.full((self.n, self.n), ' ')
        for idx in range(self.n):
            grid[idx, self.X_by_row[idx]] = 'X'
            grid[idx, self.O_by_row[idx]] = 'O'
            
        return grid

    def row_commutation_permitted(self, first_row_idx):
        if(first_row_idx < 0 or first_row_idx >= self.n-1):
            return False
        
        # Determine if the markings of one row are "sandwiched" between another or if they are disjoint
        a_l = [self.X_by_row[first_row_idx], self.O_by_row[first_row_idx]]
        a_l.sort()
        b_l = [self.X_by_row[first_row_idx+1], self.O_by_row[first_row_idx+1]]
        b_l.sort()

        # For what these variable names mean, see the row_commutation() function
        a_1 = a_l[0]
        a_2 = a_l[1]
        b_1 = b_l[0]
        b_2 = b_l[1]
        assert(a_1 < a_2)
        assert(b_1 < b_2)
        
        # The first two cases are when one is nested in another. The last two are when they are disjoint.
        if((a_1 < b_1 and a_2 > b_2) or (a_1 > b_1 and a_2 < b_2) or (a_2 < b_1) or (b_2 < a_1)):
            return True
        else: # Cannot perform it
            return False

    def column_commutation_permitted(self, first_col_idx):
        if(first_col_idx < 0 or first_col_idx >= self.n-1):
            return False
        
        # a_l contains the row indices of the letter markings in the first column
        # b_l contains the row indices of the letter markings in the second column
        a_l = [first_occurrence(self.X_by_row, first_col_idx), first_occurrence(self.O_by_row, first_col_idx)]
        b_l = [first_occurrence(self.X_by_row, first_col_idx+1), first_occurrence(self.O_by_row, first_col_idx+1)]

        # Sort them in increasing order
        a_l.sort()
        b_l.sort()
        
        a_1 = a_l[0]
        a_2 = a_l[1]
        b_1 = b_l[0]
        b_2 = b_l[1]

        if((a_1 < b_1 and a_2 > b_2) or (a_1 > b_1 and a_2 < b_2) or (a_2 < b_1) or (b_2 < a_1)):
            return True
        else: # Cannot
            return False

    def stabilization(self, row, col, letter, dir):
        stabilization_worker(self.X_by_row, self.O_by_row, row, col, letter, dir, self.n, [None, None],[None, None])
        self.n = self.n+1

    def stabilization_permitted(self, row, col, letter):
        if(letter == 1):
            if(self.X_by_row[row] == col):
                return True
            else:
                return False
        else:
            if(self.O_by_row[row] == col):
                return True
            else:
                return False

    def destabilization(self, 
                        row_coord, # Row coordinate of empty spot in square
                        col_coord, # Column coordinate of empty spot in square
                        letter, 
                        dir): # Position opposite of empty spot
       
        assert(self.X_by_row[row_coord] != col_coord and self.O_by_row[row_coord] != col_coord)

        if(letter == X):
            self.X_by_row[row_coord] = col_coord
            other_letter_dict = self.O_by_row
        else:
            self.O_by_row[row_coord] = col_coord
            other_letter_dict = self.X_by_row

        # Get the coordinates of the cell opposite to the empty cell in the 2x2 square
        if(dir == DIR_NE):
            opp_x = col_coord-1
            opp_y = row_coord+1
        elif(dir == DIR_NW):
            opp_x = col_coord+1
            opp_y = row_coord+1
        elif(dir == DIR_SE):
            opp_x = col_coord-1
            opp_y = row_coord-1
        elif(dir == DIR_SW):
            opp_x = col_coord+1
            opp_y = row_coord-1

        assert(other_letter_dict[opp_y] == opp_x)

        # Shrink the grid
        self.X_by_row[opp_y:self.n-1] = self.X_by_row[opp_y+1:self.n]
        self.X_by_row[self.n-1] = -1

        self.O_by_row[opp_y:self.n-1] = self.O_by_row[opp_y+1:self.n]
        self.O_by_row[self.n-1] = -1

        increment_if_geq(self.X_by_row, opp_x, -1)
        increment_if_geq(self.O_by_row, opp_x, -1)
        
        self.n = self.n-1 

    def Legendrian_Birth_Upper_Left_Corner(self): # Additional Legendrian elementary cobordism moves
        increment_if_geq(self.X_by_row, 0, 2)
        increment_if_geq(self.O_by_row, 0, 2)

        self.X_by_row[2:self.n+2] = self.X_by_row[0:self.n]
        self.O_by_row[2:self.n+2] = self.O_by_row[0:self.n]
        
        self.X_by_row[0] = 1
        self.X_by_row[1] = 0

        self.O_by_row[1] = 1
        self.O_by_row[0] = 0
        
        self.n += 2

    def Load_From_Diagram(self, diagram):
        a,b = diagram.shape
      
        assert(a == self.n)
        for row_idx, row in enumerate(diagram):
            for col_idx, entry in enumerate(row):
                if(entry == 'X'):
                    self.X_by_row[row_idx] = col_idx
                elif(entry == 'O'):
                    self.O_by_row[row_idx] = col_idx
                elif(entry == ' '):
                    pass
                else:
                    raise NotImplementedError

    def destabilization_permitted(self, 
                                 row_coord, # Coordinates of empty spot in 2x2 grid
                                 col_coord, 
                                 letter, # Letter that presumably appears twice in the square
                                 dir): # Direction
        
        if(row_coord < 0 or row_coord > self.n-1 or col_coord < 0 and col_coord > self.n-1):
            return False
        
        if(self.X_by_row[row_coord] == col_coord or self.O_by_row[row_coord] == col_coord):
            return False
        
        if(letter == X):
            same_letter_dict = self.X_by_row
            other_letter_dict = self.O_by_row
        elif(letter == O):
            same_letter_dict = self.O_by_row
            other_letter_dict = self.X_by_row

        
        if(dir == DIR_NE):
            # For each direction, we perform three validity checks:

            # 1: Whether the 2x2 destabilization square is too close to the boundaries
            if(row_coord == self.n-1 or col_coord == 0):
                return False
            
            # 2: Whether the two relevant corners of the 2x2 square are the correct letter
            if(not (same_letter_dict[row_coord+1] == col_coord and same_letter_dict[row_coord] == col_coord-1)):
                return False
            
            # 3: Whether there exists an empty cell in the required location
            if(other_letter_dict[row_coord+1] != col_coord-1):
                return False
            
        elif(dir == DIR_SW):
            if(row_coord == 0 or col_coord == self.n-1):
                return False
            if(not (same_letter_dict[row_coord-1] == col_coord and same_letter_dict[row_coord] == col_coord+1)):
                return False
            if(other_letter_dict[row_coord-1] != col_coord+1):
                return False
            
        elif(dir == DIR_NW):
            if(row_coord == self.n-1 or col_coord == self.n-1):
                return False
            if(not (same_letter_dict[row_coord+1] == col_coord and same_letter_dict[row_coord] == col_coord+1)):
                return False
            if(other_letter_dict[row_coord+1] != col_coord+1):
                return False
        
        else: 
            if(row_coord == 0 or col_coord == 0):
                return False
            if(not (same_letter_dict[row_coord-1] == col_coord and same_letter_dict[row_coord] == col_coord-1)):
                return False
            if(other_letter_dict[row_coord-1] != col_coord-1):
                return False
        
        return True
    
    # Perform a copinch
    def Legendrian_CoPinch_Row(self, first_row):
        assert(first_row <= self.n-2)
        
        # Get the column indices of the letter markings in each row in a sorted array
        row1_nonempty = [self.X_by_row[first_row], self.O_by_row[first_row]]
        row1_nonempty.sort()
        
        row2_nonempty = [self.X_by_row[first_row+1], self.O_by_row[first_row+1]]
        row2_nonempty.sort()

        
        if(row1_nonempty[0] > row2_nonempty[1]):

            # Make sure the corner directions are correct
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[0], self.n) == DIR_NW and 
               get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[1], self.n) == DIR_SE):
               
                if(self.X_by_row[first_row] == row1_nonempty[0]):
                    
                    # Since this function should only have been run after the Legendrian_Copinch_row_permitted function has been run on the row,
                    # we run this assert() to ensure that both of the markings in the middle are of the same letter
                    assert(self.X_by_row[first_row+1] == row2_nonempty[1])

                    # Perform the letter swap for the copinch
                    self.X_by_row[first_row], self.X_by_row[first_row+1] = self.X_by_row[first_row+1], self.X_by_row[first_row]
                
                elif(self.O_by_row[first_row] == row1_nonempty[0]):
                    assert(self.O_by_row[first_row+1] == row2_nonempty[1])
                    self.O_by_row[first_row], self.O_by_row[first_row+1] = self.O_by_row[first_row+1], self.O_by_row[first_row]
                
                else:
                    raise NotImplementedError
            else:
                raise NotImplementedError
        
        # The other case
        elif(row1_nonempty[1] > row2_nonempty[0] and row1_nonempty[0] < row2_nonempty[0] and row2_nonempty[1] > row1_nonempty[1]):
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[1], self.n) == DIR_SE and 
               get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[0], self.n) == DIR_NW):
               
                if(self.X_by_row[first_row] == row1_nonempty[1]):
                    assert(self.X_by_row[first_row+1] == row2_nonempty[0])
                    self.X_by_row[first_row], self.X_by_row[first_row+1] = self.X_by_row[first_row+1], self.X_by_row[first_row]
                
                elif(self.O_by_row[first_row] == row1_nonempty[1]):
                    assert(self.O_by_row[first_row+1] == row2_nonempty[0])
                    self.O_by_row[first_row], self.O_by_row[first_row+1] = self.O_by_row[first_row+1], self.O_by_row[first_row]
                
                else:
                    raise NotImplementedError
            else:
                raise NotImplementedError
        else:
            raise NotImplementedError
        
    def Legendrian_CoPinch_Row_Permitted(self, first_row):
        if(first_row > self.n-2):
            return False
        row1_nonempty = [self.X_by_row[first_row], self.O_by_row[first_row]]
        row1_nonempty.sort()
        
        row2_nonempty = [self.X_by_row[first_row+1], self.O_by_row[first_row+1]]
        row2_nonempty.sort()

        if(row1_nonempty[0] > row2_nonempty[1]):
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[0], self.n) == DIR_NW and get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[1], self.n) == DIR_SE):
                if((self.X_by_row[first_row] == row1_nonempty[0] and self.X_by_row[first_row+1] == row2_nonempty[1]) or (self.O_by_row[first_row] == row1_nonempty[0] and self.O_by_row[first_row+1] == row2_nonempty[1])):
                    for col_idx in range(row2_nonempty[1]+1, row1_nonempty[0]):
                        col_nonempty = [first_occurrence(self.X_by_row, col_idx), first_occurrence(self.O_by_row, col_idx)]
                        assert(-1 not in col_nonempty)
                        col_nonempty.sort()

                        if(not (col_nonempty[0] > first_row+1 or col_nonempty[1] < first_row)):
                            return False
                    
                    return True

        elif(row1_nonempty[1] > row2_nonempty[0] and row1_nonempty[0] < row2_nonempty[0] and row2_nonempty[1] > row1_nonempty[1]):
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[1], self.n) == DIR_SE and get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[0], self.n) == DIR_NW):
                
                if((self.X_by_row[first_row] == row1_nonempty[1] and self.X_by_row[first_row+1] == row2_nonempty[0]) or (self.O_by_row[first_row] == row1_nonempty[1] and self.O_by_row[first_row+1] == row2_nonempty[0])):    
                    for col_idx in range(row2_nonempty[0]+1, row1_nonempty[1]):
                        col_nonempty = [first_occurrence(self.X_by_row, col_idx), first_occurrence(self.O_by_row, col_idx)]
                        assert(-1 not in col_nonempty)
                        col_nonempty.sort()

                        if(not (col_nonempty[0] > first_row+1 or col_nonempty[1] < first_row)):
                            return False
                    
                    return True
        
        return False

    def Legendrian_Pinch_Row(self, first_row):

        if(first_row > self.n-2):
            return False
        
        row1_nonempty = [self.X_by_row[first_row], self.O_by_row[first_row]]
        row1_nonempty.sort()
        
        row2_nonempty = [self.X_by_row[first_row+1], self.O_by_row[first_row+1]]
        row2_nonempty.sort()
        
        if(row1_nonempty[0] < row2_nonempty[1] and row1_nonempty[1] > row2_nonempty[1] and row1_nonempty[0] > row2_nonempty[0]):
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[0], self.n) == DIR_SW and get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[1], self.n) == DIR_NE):
                if(self.X_by_row[first_row] == row1_nonempty[0]):
                    assert(self.X_by_row[first_row+1] == row2_nonempty[1])
                    self.X_by_row[first_row], self.X_by_row[first_row+1] = self.X_by_row[first_row+1], self.X_by_row[first_row]
                elif(self.O_by_row[first_row] == row1_nonempty[0]):
                    assert(self.O_by_row[first_row+1] == row2_nonempty[1])
                    self.O_by_row[first_row], self.O_by_row[first_row+1] = self.O_by_row[first_row+1], self.O_by_row[first_row]
                else:
                    raise NotImplementedError
            else:
                raise NotImplementedError
        
        elif(row1_nonempty[1] < row2_nonempty[0]):
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[1], self.n) == DIR_NE and get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[0], self.n) == DIR_SW):
                if(self.X_by_row[first_row] == row1_nonempty[1]):
                    assert(self.X_by_row[first_row+1] == row2_nonempty[0])
                    self.X_by_row[first_row], self.X_by_row[first_row+1] = self.X_by_row[first_row+1], self.X_by_row[first_row]
                elif(self.O_by_row[first_row] == row1_nonempty[1]):
                    assert(self.O_by_row[first_row+1] == row2_nonempty[0])
                    self.O_by_row[first_row], self.O_by_row[first_row+1] = self.O_by_row[first_row+1], self.O_by_row[first_row]
                else:
                    raise NotImplementedError
                
                
        else:
            raise NotImplementedError

    # Check to see if pinching is permitted for a row
    def Legendrian_Pinch_Row_Permitted(self, 
                                       first_row # The index of the first of the two rows. The second row is the row directly below this one
                                       ):
        
        # Ensure first_row is not the very last row
        if(first_row > self.n-2):
            return False
        

        row1_nonempty = [self.X_by_row[first_row], self.O_by_row[first_row]]
        row1_nonempty.sort()
        
        row2_nonempty = [self.X_by_row[first_row+1], self.O_by_row[first_row+1]]
        row2_nonempty.sort()
        
        if(row1_nonempty[0] < row2_nonempty[1] and row1_nonempty[1] > row2_nonempty[1] and row1_nonempty[0] > row2_nonempty[0]):
            
            # check Corners are valid
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[0], self.n) == DIR_SW and 
               get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[1], self.n) == DIR_NE):
               
                # Make sure the two middle markings are the same letter
                if((self.X_by_row[first_row] == row1_nonempty[0] and self.X_by_row[first_row+1] == row2_nonempty[1]) or 
                   (self.O_by_row[first_row] == row1_nonempty[0] and self.O_by_row[first_row+1] == row2_nonempty[1])):
                    
                    # Make sure there are no crossing obstructions between the two middle letters 
                    for col_idx in range(row1_nonempty[0]+1, row2_nonempty[1]):
                        col_nonempty = [first_occurrence(self.X_by_row, col_idx), first_occurrence(self.O_by_row, col_idx)]
                        col_nonempty.sort()

                        if(not (col_nonempty[0] > first_row+1 or col_nonempty[1] < first_row)):
                            return False
                        
                    return True
        elif(row1_nonempty[1] < row2_nonempty[0]):
            
            # Check corners are valid
            if(get_corner_dir(self.X_by_row, self.O_by_row, first_row, row1_nonempty[1], self.n) == DIR_NE and 
               get_corner_dir(self.X_by_row, self.O_by_row, first_row+1, row2_nonempty[0], self.n) == DIR_SW):
                
                # Make sure the two middle markings are the same letter
                if((self.X_by_row[first_row] == row1_nonempty[1] and self.X_by_row[first_row+1] == row2_nonempty[0]) or 
                   (self.O_by_row[first_row] == row1_nonempty[1] and self.O_by_row[first_row+1] == row2_nonempty[0])):

                    # Make sure there are no crossing obstructions between the two middle letters 
                    for col_idx in range(row1_nonempty[1]+1, row2_nonempty[0]):
                        col_nonempty = [first_occurrence(self.X_by_row, col_idx), first_occurrence(self.O_by_row, col_idx)]
                        col_nonempty.sort()

                        if(not (col_nonempty[0] > first_row+1 or col_nonempty[1] < first_row)):
                            return False

                    return True
                    
        return False
    
    def Legendrian_Death(self, 
                         row_idx,    # row and column of top left position of the 2x2 unknot
                         col_idx):
        if(self.n <= 3):
            raise NotImplementedError
        
        if(row_idx < 0 or row_idx >= self.n-1 or col_idx < 0 or col_idx >= self.n-1):
            raise NotImplementedError
        
        '''
            Make sure it either looks like
            XO
            OX

            or

            OX
            XO
        '''
        if((self.X_by_row[row_idx] == col_idx and self.X_by_row[row_idx+1] == col_idx+1 and self.O_by_row[row_idx] == col_idx+1 and self.O_by_row[row_idx+1] == col_idx)
           or
           (self.O_by_row[row_idx] == col_idx and self.O_by_row[row_idx+1] == col_idx+1 and self.X_by_row[row_idx] == col_idx+1 and self.X_by_row[row_idx+1] == col_idx)
           ):
            self.X_by_row[row_idx:self.n-2] = self.X_by_row[row_idx+2:self.n]
            self.O_by_row[row_idx:self.n-2] = self.O_by_row[row_idx+2:self.n]
            self.X_by_row[self.n-2] = -1
            self.X_by_row[self.n-1] = -1
            self.O_by_row[self.n-2] = -1
            self.O_by_row[self.n-1] = -1
            
            self.n = self.n - 2
            increment_if_geq(self.X_by_row, col_idx+2, -2)
            increment_if_geq(self.O_by_row, col_idx+2, -2)
            
        else:
            raise NotImplementedError
       
    def Legendrian_Death_Permitted(self, 
                         row_idx,    # row and column of top left position of the "loop"
                         col_idx):
        if(self.n <= 3):
            return False
        
        if(row_idx < 0 or row_idx >= self.n-1 or col_idx < 0 or col_idx >= self.n-1):
            return False
        
        if((self.X_by_row[row_idx] == col_idx and self.X_by_row[row_idx+1] == col_idx+1 and self.O_by_row[row_idx] == col_idx+1 and self.O_by_row[row_idx+1] == col_idx)
           or
           (self.O_by_row[row_idx] == col_idx and self.O_by_row[row_idx+1] == col_idx+1 and self.X_by_row[row_idx] == col_idx+1 and self.X_by_row[row_idx+1] == col_idx)
           ):
            return True
            
        else:
            return False
    
    def copy_gd(self): # Returns copy of grid diagram
        new_gd = GridDiagram(self.n)
        new_gd.X_by_row = self.X_by_row.copy()
        new_gd.O_by_row = self.O_by_row.copy()
    
        return new_gd
    
    # Get integer hash of a grid
    def hash(self):
        return hash((self.X_by_row.tobytes(), self.O_by_row.tobytes(), self.n))
    


In [4]:
class KnotWrapper:
    def __init__(self, gd=None):
        self.components = 0
        self.move_history = [] # Store moves in verbal form (e.g. "pinch row 4 with row 5")
        self.grid_history = [] # Store moves in grid form

        # Count the number of moves made so far
        self.n_births = 0 
        self.n_deaths = 0
        self.n_stabilizations = 0
        self.n_destabilizations = 0
        self.n_pinches = 0
        self.n_copinches = 0
        self.n_commutations = 0

        if(gd != None):
            self.load_gd(gd)
    
    def validate_gd(self):
        return self.gd.validate()
        
    def load_gd(self, gd):
        self.gd = gd
        self.grid_history.append((gd.X_by_row, gd.O_by_row))
        self.n = gd.n
        assert(self.n >= 2)
        
    def load_move_history(self, prev_kw):
        for old_history_item in prev_kw.move_history:
            self.move_history.append(old_history_item)
            
        self.n_births = prev_kw.n_births
        self.n_deaths = prev_kw.n_deaths
        self.n_stabilizations = prev_kw.n_stabilizations
        self.n_destabilizations = prev_kw.n_destabilizations
        self.n_pinches = prev_kw.n_pinches
        self.n_copinches = prev_kw.n_copinches
        self.n_commutations = prev_kw.n_commutations

    def load_grid_history(self, prev_kw):
        for old_grid_item in prev_kw.grid_history:
            self.grid_history.append(old_grid_item)

    def add_move(self, move):
        self.move_history.append(move)

    def add_grid_history(self, grid):
        self.grid_history.append(grid)

    def hash(self):
        return self.gd.hash()

# Create a new Knotwrapper object after performing a grid move
def produce_new_kw(prev_gd, hist_kw, message,
                   births=0,
                   deaths=0,
                   stabilizations=0,
                   destabilizations=0,
                   pinches=0,
                   copinches=0,
                   commutations=0,
                   ):
    
    new_kw = KnotWrapper()
    
    new_kw.load_grid_history(hist_kw)
    new_kw.load_move_history(hist_kw)
    new_kw.load_gd(prev_gd)

    # Append the new move's message to the move history
    new_kw.add_move(message)
    
    new_kw.n_births += births
    new_kw.n_deaths += deaths
    new_kw.n_stabilizations += stabilizations
    new_kw.n_destabilizations += destabilizations
    new_kw.n_pinches += pinches
    new_kw.n_copinches += copinches
    new_kw.n_commutations += commutations
    
    return new_kw

In [5]:


# Get possible grid moves to be performed on a grid 
def possible_moves(gd: GridDiagram, 
                   
                   # Optional flags
                   get_pinches=True, 
                   get_copinches=True, 
                   get_deaths=True):

    ok_row_commutations = []
    ok_col_commutations = []
    ok_row_pinches = []
    ok_row_copinches = []

    ok_death_moves = []
    ok_stab = []
    ok_destab = []

    # The return array
    ret = []

    for row_idx in range(gd.n):
        if(gd.row_commutation_permitted(row_idx)):
            ok_row_commutations.append(row_idx)
            
    if(get_pinches):
        for row_idx in range(gd.n):
            if(gd.Legendrian_Pinch_Row_Permitted(row_idx)):
                ok_row_pinches.append(row_idx)

    if(get_copinches):
        for row_idx in range(gd.n):
            if(gd.Legendrian_CoPinch_Row_Permitted(row_idx)):
                ok_row_copinches.append(row_idx)
    
    for col_idx in range(gd.n):
        if(gd.column_commutation_permitted(col_idx)):
            ok_col_commutations.append(col_idx)

    if(get_deaths and gd.n > 2):
        for row in range(gd.n):
            col = gd.X_by_row[row]
            if(gd.Legendrian_Death_Permitted(row, col)):
                ok_death_moves.append((row, col))
        for row in range(gd.n):
            col = gd.O_by_row[row]
            if(gd.Legendrian_Death_Permitted(row, col)):
                ok_death_moves.append((row, col))
        
    for row in range(gd.n):
        col = gd.X_by_row[row]
        ok_stab.append((row, col, DIR_NE, 1))
        ok_stab.append((row, col, DIR_SW, 1))
    
    for row in range(gd.n):
        col = gd.O_by_row[row]
        ok_stab.append((row, col, DIR_NE, 2))
        ok_stab.append((row, col, DIR_SW, 2))
   

    for row in range(gd.n):
        col = gd.X_by_row[row]
        if(gd.destabilization_permitted(row, col+1, letter=1, dir=DIR_NE)):
            ok_destab.append((row, col+1, DIR_NE, 1))
   
    for row in range(gd.n):
        col = gd.X_by_row[row]
        if(gd.destabilization_permitted(row+1, col, letter=1, dir=DIR_SW)):
            ok_destab.append((row+1, col, DIR_SW, 1))
    
    for row in range(gd.n):
        col = gd.O_by_row[row]
        if(gd.destabilization_permitted(row, col+1, letter=2, dir=DIR_NE)):
            ok_destab.append((row, col+1, DIR_NE, 2))
   
    for row in range(gd.n):
        col = gd.O_by_row[row]
        if(gd.destabilization_permitted(row+1, col, letter=2, dir=DIR_SW)):
            ok_destab.append((row+1, col, DIR_SW, 2))
    

    ret = [ok_row_commutations, ok_col_commutations]
    if(get_pinches):
        ret.append(ok_row_pinches)
    if(get_copinches):
        ret.append(ok_row_copinches)
    if(get_deaths):
        ret.append(ok_death_moves)
    ret += [ok_stab, ok_destab]

    
    return ret

def search(L_minus,
           L_plus,
           tree_depth,

           # Default Search parameters
           max_grid_size=8,
           max_births=2,
           max_deaths=2,
           max_stabilizations=100000,
           max_destabilizations=100000,
           max_pinches=100000,
           max_copinches=100000,
           max_commutations=100000,
           isotopy_only = False,

           # Default log file 
           logfile='log.txt'
           ):
  

    if(isotopy_only):
        max_births = 0
        max_deaths = 0
        max_pinches = 0
        max_copinches = 0

    set_logfile(logfile)

    minus_tb = L_minus.gd.tb()
    plus_tb = L_plus.gd.tb()

    minus_r = L_minus.gd.r()
    plus_r = L_plus.gd.r()

    # 0 will be the "negative" side; "1" will be the positive side.
    tree_minus_seen = {} # Format will be "hashvalue" : [list of unique knot descriptors]
    tree_plus_seen = {} # Same

    tree_minus_current = [L_minus]
    tree_plus_current = [L_plus]
    tree_minus_seen[L_minus.hash()] = L_minus
    tree_plus_seen[L_plus.hash()] = L_plus
    
    t_minus_births=0
    t_minus_stabilizations=0
    t_minus_destabilizations=0
    t_minus_copinches=0
    t_minus_r_commutations=0 
    t_minus_c_commutations=0

    t_plus_deaths=0
    t_plus_stabilizations=0
    t_plus_destabilizations=0
    t_plus_pinches=0
    t_plus_r_commutations=0
    t_plus_c_commutations=0

    found = False

    log("starting")
    param_str = f"""tree_depth={tree_depth}, \n
           max_grid_size={max_grid_size}, \n
           max_births={max_births}, \n
           max_deaths={max_deaths}, \n
           max_stabilizations={max_stabilizations}, \n
           max_destabilizations={max_destabilizations}, \n
           max_pinches={max_pinches}, \n
           max_copinches={max_copinches}, \n
           max_commutations={max_commutations}, \n
           isotopy_only={isotopy_only}"""
    log(param_str)
    log("Minus direction:\n")
    log("\n" + str(L_minus.gd.convert_to_grid()))
    log(f"(tb,r) = ({minus_tb},{minus_r})")
    log("Positive direction:\n")
    log("\n" + str(L_plus.gd.convert_to_grid()))
    log(f"(tb,r) = ({plus_tb},{plus_r})")

    for level in range(tree_depth):
        log(f'At level {level}. At this level, there are {len(tree_minus_current)} diagrams to be checked from the first tree and {len(tree_plus_current)} diagrams to be checked from the second tree.')
        log(f'{len(tree_minus_seen)} diagrams have been seen in the first tree in total')
        log(f'{len(tree_plus_seen)} diagrams have been seen in the second tree in total')
        
        tree_minus_next = []
        tree_plus_next = []

        for index, knotwrapper in enumerate(tree_minus_current):
            if(index % 10000 == 0):
                log(f"Processing {index}/{len(tree_minus_current)}")
            
            # Get possible moves
            ok_row_commutations, ok_col_commutations, ok_row_copinches, ok_death_moves, ok_stab, ok_destab = possible_moves(knotwrapper.gd, get_pinches=False)
          
            if(knotwrapper.n_commutations < max_commutations):
                for row_idx in ok_row_commutations:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    row_commutation(candidate_gd.X_by_row, candidate_gd.O_by_row, row_idx, candidate_gd.n)

                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_minus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'commuted row {row_idx} with row {row_idx+1}', commutations=1)
                        tree_minus_next.append(new_kw)
                        tree_minus_seen[gd_hash] = new_kw
                        t_minus_r_commutations += 1

                        if(gd_hash in tree_plus_seen):
                            found = True
                            break

                for col_idx in ok_col_commutations:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    column_commutation(candidate_gd.X_by_row, candidate_gd.O_by_row, col_idx, candidate_gd.n)

                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_minus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'commuted col {col_idx} with col {col_idx+1}', commutations=1)
                        tree_minus_next.append(new_kw)
                        tree_minus_seen[gd_hash] = new_kw
                        t_minus_c_commutations += 1
                        
                        if(gd_hash in tree_plus_seen):
                            found = True
                            break
                        
            if(knotwrapper.n_copinches < max_copinches):   
                for row_idx in ok_row_copinches:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    candidate_gd.Legendrian_CoPinch_Row(row_idx)
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_minus_seen):
                        
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'copinched row {row_idx} with row {row_idx+1}', copinches=1)
                        tree_minus_seen[gd_hash] = new_kw

                    if not(candidate_gd.count_components() == 1 and candidate_gd.r() != plus_r):
                        tree_minus_next.append(new_kw)
                        t_minus_copinches += 1

                        if(gd_hash in tree_plus_seen):
                            found = True
                            break
           
            # Legendrian birth in upper left corner:
            if(knotwrapper.n_births < max_births and knotwrapper.n < max_grid_size):
                candidate_gd = knotwrapper.gd.copy_gd()
                candidate_gd.Legendrian_Birth_Upper_Left_Corner()
                
                gd_hash = candidate_gd.hash()
                if(gd_hash not in tree_minus_seen):
                    new_kw = produce_new_kw(candidate_gd, knotwrapper, f'Birthed an unknot in the upper left corner', births=1)
                    tree_minus_next.append(new_kw)
                    tree_minus_seen[gd_hash] = new_kw

                    t_minus_births += 1

                    if(gd_hash in tree_plus_seen):
                        found = True
                        break
                      

            if(knotwrapper.n_stabilizations < max_stabilizations and knotwrapper.n < max_grid_size):
                for row_idx, col_idx, dir_name_idx, the_letter in ok_stab:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    
                    candidate_gd.stabilization(row_idx, col_idx, the_letter, dir_name_idx)
                    
                    assert(candidate_gd.n <= max_grid_size)
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_minus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f"Performed {the_letter}:{dirs_names[dir_name_idx]} stabilization at {(row_idx, col_idx)}", stabilizations=1)
                        tree_minus_next.append(new_kw)
                        tree_minus_seen[gd_hash] = new_kw
                        t_minus_stabilizations += 1

                        if(gd_hash in tree_plus_seen):
                            found = True
                            break
                        

            if(knotwrapper.n_destabilizations < max_destabilizations):
                for row_idx, col_idx, dir_name_idx, the_letter in ok_destab:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    candidate_gd.destabilization(row_idx, col_idx, the_letter, dir_name_idx)

                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_minus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f"Performed {the_letter}:{dirs_names[dir_name_idx]} destabilization at {(row_idx, col_idx)}", destabilizations=1)
                        tree_minus_next.append(new_kw)
                        tree_minus_seen[gd_hash] = new_kw
                        
                        t_minus_destabilizations += 1
                        if(gd_hash in tree_plus_seen):
                            found = True
                            break

    
        for index, knotwrapper in enumerate(tree_plus_current):
            if(found):
                break

            if(index % 10000 == 0):
                log(f"Processing {index}/{len(tree_plus_current)}")
            ok_row_commutations, ok_col_commutations, ok_row_pinches, ok_death_moves, ok_stab, ok_destab = possible_moves(knotwrapper.gd, get_copinches=False)
            
            if(knotwrapper.n_commutations < max_commutations):
                for row_idx in ok_row_commutations:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    row_commutation(candidate_gd.X_by_row, candidate_gd.O_by_row, row_idx, candidate_gd.n)
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_plus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'commuted row {row_idx} with row {row_idx+1}', commutations=1)
                        tree_plus_next.append(new_kw)
                        tree_plus_seen[gd_hash] = new_kw

                        t_plus_r_commutations += 1
                        if(gd_hash in tree_minus_seen):
                            found = True
                            break
                        
                for col_idx in ok_col_commutations:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    column_commutation(candidate_gd.X_by_row, candidate_gd.O_by_row, col_idx, candidate_gd.n)
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_plus_seen):             
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'commuted col {col_idx} with col {col_idx+1}', commutations=1)
                        tree_plus_next.append(new_kw)
                        tree_plus_seen[gd_hash] = new_kw

                        t_plus_c_commutations += 1
                        if(gd_hash in tree_minus_seen):
                            found = True
                            break
       
            if(knotwrapper.n_pinches < max_pinches):         
                for row_idx in ok_row_pinches:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    candidate_gd.Legendrian_Pinch_Row(row_idx)
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_plus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'pinched row {row_idx} with row {row_idx+1}', pinches=1)
                        tree_plus_seen[gd_hash] = new_kw

                        components = candidate_gd.count_components()
                        tb_diff = minus_tb - candidate_gd.tb()
                        if (not (components == 1 and candidate_gd.r() != minus_r)) and not(tb_diff >= components):
                            tree_plus_next.append(new_kw)
                            
                            t_plus_pinches += 1
                            if(gd_hash in tree_minus_seen):
                                found = True
                                break
        
      
            # Death moves allowed from + dir
            if(knotwrapper.n_deaths < max_deaths):
                for (row_idx, col_idx) in ok_death_moves:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    candidate_gd.Legendrian_Death(row_idx, col_idx)
                    
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_plus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'a link at {(row_idx, col_idx)} died.', deaths=1)
                        tree_plus_next.append(new_kw)
                        tree_plus_seen[gd_hash] = new_kw

                        t_plus_deaths += 1
                        if(gd_hash in tree_minus_seen):
                            found = True
                            break
            

            if(knotwrapper.n_stabilizations < max_stabilizations and knotwrapper.n < max_grid_size):
                for row_idx, col_idx, dir_name_idx, the_letter in ok_stab:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    candidate_gd.stabilization(row_idx, col_idx, the_letter, dir_name_idx)
                    assert(candidate_gd.n <= max_grid_size)
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_plus_seen):
                        
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'Performed {the_letter}:{dirs_names[dir_name_idx]} stabilization at {(row_idx, col_idx)}', stabilizations=1)
                        tree_plus_next.append(new_kw)
                        tree_plus_seen[gd_hash] = new_kw

                        t_plus_stabilizations += 1
                        if(gd_hash in tree_minus_seen):
                            found = True
                            break

            if(knotwrapper.n_destabilizations < max_destabilizations):
                for row_idx, col_idx, dir_name_idx, the_letter in ok_destab:
                    candidate_gd = knotwrapper.gd.copy_gd()
                    candidate_gd.destabilization(row_idx, col_idx, the_letter, dir_name_idx)
                    
                    gd_hash = candidate_gd.hash()
                    if(gd_hash not in tree_plus_seen):
                        new_kw = produce_new_kw(candidate_gd, knotwrapper, f'Performed {the_letter}:{dirs_names[dir_name_idx]} destabilization at {(row_idx, col_idx)}', destabilizations=1)
                        tree_plus_next.append(new_kw)
                        tree_plus_seen[gd_hash] = new_kw
                        t_plus_destabilizations += 1

                        if(gd_hash in tree_minus_seen):
                            found = True
                            break
  

        
        log(f't_minus:\nbirths: {t_minus_births}\nstabilizations: {t_minus_stabilizations}\ndestabilizations: {t_minus_destabilizations}\ncopinches: {t_minus_copinches}\ncommutations: {(t_minus_r_commutations,t_minus_c_commutations)}')
        log(f't_plus:\ndeaths: {t_plus_deaths}\nstabilizations: {t_plus_stabilizations}\ndestabilizations: {t_plus_destabilizations}\npinches: {t_plus_pinches}\ncommutations: {(t_plus_r_commutations,t_plus_c_commutations)}')
      
        intersecting = set(tree_minus_seen).intersection(set(tree_plus_seen))
        if(len(intersecting) > 0): # We've found one
            log("Found:")
            break
        
        tree_minus_current = []
        tree_plus_current = []
        for item in tree_minus_next:
            tree_minus_current.append(item)
        for item in tree_plus_next:
            tree_plus_current.append(item)

    if(len(intersecting) >= 1):
        log('The move history: ')
        hashvalue = list(intersecting)[0]
        log('Move history from the MINUS (-) direction:\n')
        for item in tree_minus_seen[hashvalue].move_history:
            log(str(item))

        log('Move history from the PLUS (+) direction:\n')
        for item in tree_plus_seen[hashvalue].move_history:
            log(str(item))
        

        log('Grid history from the MINUS (-) direction:\n')
        for item_idx, item in enumerate(tree_minus_seen[hashvalue].grid_history):
            log("\n" + str(marking_tuple_to_grid(item)))
            if(item_idx <= len(tree_minus_seen[hashvalue].move_history)-1):
                log("\n" + str(tree_minus_seen[hashvalue].move_history[item_idx]))
       
        log('Grid history from the PLUS (+) direction:\n')
        for item_idx, item in enumerate(reversed(tree_plus_seen[hashvalue].grid_history)):
            log("\n" + str(marking_tuple_to_grid(item)))
            if(len(tree_plus_seen[hashvalue].move_history) - item_idx >= 0 and (len(tree_plus_seen[hashvalue].move_history) - item_idx -1 >= 0)):
                log("\n" + str(tree_plus_seen[hashvalue].move_history[len(tree_plus_seen[hashvalue].move_history) - item_idx -1]))
              
    return tree_minus_seen, tree_plus_seen




In [6]:


now = datetime.now() # current date and time
date_time = now.strftime("%m_%d_%Y_%H_%M_%S")

gd1 = GridDiagram(gridsize=9)
gd1.Load_From_Diagram(grid_m_10_132)

gd2 = GridDiagram(gridsize=9)
gd2.Load_From_Diagram(grid_m_10_145_1)

Lambda_minus = KnotWrapper(gd=gd1)
Lambda_plus = KnotWrapper(gd=gd2)

t_minus, t_plus = search(Lambda_minus, 
                Lambda_plus, 
                tree_depth=70,
                max_grid_size=12,
                max_births=1,
                max_deaths=5, 
                logfile=f"m(10_132) to m(10_145)_1 log{date_time}.txt",
                isotopy_only=False)


09/03/2024, 11:13:44: starting
09/03/2024, 11:13:44: tree_depth=70, 

           max_grid_size=12, 

           max_births=1, 

           max_deaths=5, 

           max_stabilizations=100000, 

           max_destabilizations=100000, 

           max_pinches=100000, 

           max_copinches=100000, 

           max_commutations=100000, 

           isotopy_only=False
09/03/2024, 11:13:44: Minus direction:

09/03/2024, 11:13:44: 
[['O' ' ' 'X' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' 'O' ' ' 'X' ' ' ' ' ' ' ' ' ' ']
 [' ' ' ' 'O' ' ' ' ' ' ' ' ' 'X' ' ']
 ['X' ' ' ' ' ' ' ' ' ' ' 'O' ' ' ' ']
 [' ' ' ' ' ' ' ' ' ' 'X' ' ' ' ' 'O']
 [' ' ' ' ' ' ' ' 'X' ' ' ' ' 'O' ' ']
 [' ' ' ' ' ' 'O' ' ' ' ' 'X' ' ' ' ']
 [' ' 'X' ' ' ' ' ' ' 'O' ' ' ' ' ' ']
 [' ' ' ' ' ' ' ' 'O' ' ' ' ' ' ' 'X']]
09/03/2024, 11:13:44: (tb,r) = (-1.0,0.0)
09/03/2024, 11:13:44: Positive direction:

09/03/2024, 11:13:44: 
[['O' ' ' 'X' ' ' ' ' ' ' ' ' ' ' ' ']
 [' ' 'O' ' ' ' ' 'X' ' ' ' ' ' ' ' ']
 [' ' ' ' 'O' ' ' ' ' ' ' 

KeyboardInterrupt: 